# Brain-to-Audio Reconstruction Evaluation

In [60]:
import os
import re
import numpy as np
import torch
import torchaudio
import torchaudio.functional as AF
import librosa
import librosa.display
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display, Audio, HTML
from pathlib import Path
from scipy.spatial.distance import cosine
from scipy.stats import pearsonr

In [ ]:

GT_DIR   = ".../data/decoded_ip2"       # ground truth
PRED_DIR = ".../data/decoded_ip2"    

TARGET_SR   = 44100   
TOP_N       = 10      

# 'cosine_melspec', 'pearson_melspec', 'cosine_mfcc'
METRICS = ['cosine_melspec', 'cosine_mfcc']

In [ ]:
# ──  gt / pred ────────────────────────────────────────────────
def parse_sound_sample(filename):
    """Estrae (sound_idx, sample_idx) dal nome file gt o pred."""
    m = re.search(r'sound(\d+)_sample(\d+)', filename)
    if m:
        return int(m.group(1)), int(m.group(2))
    return None, None

gt_files   = {parse_sound_sample(f): os.path.join(GT_DIR, f)
              for f in os.listdir(GT_DIR) if f.endswith('.wav') and f.startswith('gt_')}
pred_files = {parse_sound_sample(f): os.path.join(PRED_DIR, f)
              for f in os.listdir(PRED_DIR) if f.endswith('.wav') and f.startswith('pred_')}


common_keys = sorted(set(gt_files) & set(pred_files))
print(f"find: {len(common_keys)}")
print(f"only in GT:   {len(set(gt_files) - set(pred_files))}")
print(f"only in PRED: {len(set(pred_files) - set(gt_files))}")

In [ ]:
def load_wav(path, target_sr=TARGET_SR):
    wav, sr = torchaudio.load(path)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != target_sr:
        wav = AF.resample(wav, sr, target_sr)
    return wav.squeeze(0).numpy()  # [samples]

def melspectrogram_features(wav, sr=TARGET_SR, n_mels=128, n_fft=1024, hop=256):
    mel = librosa.feature.melspectrogram(y=wav, sr=sr, n_mels=n_mels,
                                          n_fft=n_fft, hop_length=hop)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return mel_db.flatten()

def mfcc_features(wav, sr=TARGET_SR, n_mfcc=40):
    mfcc = librosa.feature.mfcc(y=wav, sr=sr, n_mfcc=n_mfcc)
    return mfcc.flatten()

def cosine_sim(a, b):
    min_len = min(len(a), len(b))
    a, b = a[:min_len], b[:min_len]
    return 1.0 - cosine(a, b)

def pearson_sim(a, b):
    min_len = min(len(a), len(b))
    a, b = a[:min_len], b[:min_len]
    r, _ = pearsonr(a, b)
    return float(r)

METRIC_FNS = {
    'cosine_melspec':  lambda g, p: cosine_sim(melspectrogram_features(g), melspectrogram_features(p)),
    'cosine_mfcc':     lambda g, p: cosine_sim(mfcc_features(g), mfcc_features(p)),
    'pearson_melspec': lambda g, p: pearson_sim(melspectrogram_features(g), melspectrogram_features(p)),
}

In [ ]:

results = []
for i, key in enumerate(common_keys):
    sound_idx, sample_idx = key
    gt_wav   = load_wav(gt_files[key])
    pred_wav = load_wav(pred_files[key])

    scores = {}
    for metric in METRICS:
        scores[metric] = METRIC_FNS[metric](gt_wav, pred_wav)

    avg_score = np.mean(list(scores.values()))
    results.append({
        'key':        key,
        'sound_idx':  sound_idx,
        'sample_idx': sample_idx,
        'gt_path':    gt_files[key],
        'pred_path':  pred_files[key],
        'scores':     scores,
        'avg_score':  avg_score,
    })
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(common_keys)} ...")

results.sort(key=lambda x: x['avg_score'], reverse=True)

In [ ]:

print(f"{'Rank':<5} {'Sound':>6} {'Sample':>7} ", end='')
for m in METRICS:
    print(f"{m:>20}", end='')
print(f"{'AVG':>8}")
print('-' * (5 + 7 + 8 + 20 * len(METRICS) + 8))

for rank, r in enumerate(results):
    print(f"{rank+1:<5} {r['sound_idx']:>6} {r['sample_idx']:>7} ", end='')
    for m in METRICS:
        print(f"{r['scores'][m]:>20.4f}", end='')
    print(f"{r['avg_score']:>8.4f}")

In [ ]:

avg_scores = [r['avg_score'] for r in results]

fig, axes = plt.subplots(1, len(METRICS) + 1, figsize=(5 * (len(METRICS) + 1), 4))
for ax, metric in zip(axes[:-1], METRICS):
    vals = [r['scores'][metric] for r in results]
    ax.hist(vals, bins=20, color='steelblue', edgecolor='white')
    ax.set_title(metric)
    ax.set_xlabel('Similarity')
    ax.axvline(np.mean(vals), color='tomato', linestyle='--', label=f'mean={np.mean(vals):.3f}')
    ax.legend(fontsize=8)

axes[-1].hist(avg_scores, bins=20, color='mediumseagreen', edgecolor='white')
axes[-1].set_title('Average similarity')
axes[-1].set_xlabel('Similarity')
axes[-1].axvline(np.mean(avg_scores), color='tomato', linestyle='--',
                  label=f'mean={np.mean(avg_scores):.3f}')
axes[-1].legend(fontsize=8)

plt.suptitle('Similarity GT vs Pred', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"\nSimilarity average: {np.mean(avg_scores):.4f}")
print(f"Median:                       {np.median(avg_scores):.4f}")
print(f"Max:                           {np.max(avg_scores):.4f}")
print(f"Min:                           {np.min(avg_scores):.4f}")

In [ ]:

SAMPLE_IDX = 21    # speech: 5, 2, 7, 9, 11
                  # voice: 14, 15, 17, 18, 21
                  # animal: 30, 31, 32, 35
                  # music: 40, 41, 43, 44, 45, 47
                  # nature: 48, 50, 55, 58, 59
                  # tool: 64, 66, 67, 71

def plot_and_play(sample_idx, sr=TARGET_SR):
    matches = [r for r in results if r['sample_idx'] == sample_idx]
    if not matches:
        available = sorted(set(r['sample_idx'] for r in results))
        print(f"Sample {sample_idx} not found. Available: {available}")
        return

    r = matches[0]
    gt_wav   = load_wav(r['gt_path'])
    pred_wav = load_wav(r['pred_path'])

    fig = plt.figure(figsize=(14, 4))
    gs  = gridspec.GridSpec(2, 2, figure=fig)
    ax_gt_w   = fig.add_subplot(gs[0, 0])
    ax_pred_w = fig.add_subplot(gs[0, 1])
    ax_gt_m   = fig.add_subplot(gs[1, 0])
    ax_pred_m = fig.add_subplot(gs[1, 1])

    t  = np.linspace(0, len(gt_wav)   / sr, len(gt_wav))
    t2 = np.linspace(0, len(pred_wav) / sr, len(pred_wav))

    ax_gt_w.plot(t,  gt_wav,   color='steelblue', linewidth=0.5)
    ax_gt_w.set_title('Ground Truth — waveform', fontsize=9)
    ax_gt_w.set_xlabel('s'); ax_gt_w.set_ylim(-1.05, 1.05)

    ax_pred_w.plot(t2, pred_wav, color='tomato', linewidth=0.5)
    ax_pred_w.set_title('Reconstruction — waveform', fontsize=9)
    ax_pred_w.set_xlabel('s'); ax_pred_w.set_ylim(-1.05, 1.05)

    mel_gt   = librosa.feature.melspectrogram(y=gt_wav,   sr=sr, n_mels=64, hop_length=256)
    mel_pred = librosa.feature.melspectrogram(y=pred_wav, sr=sr, n_mels=64, hop_length=256)
    librosa.display.specshow(librosa.power_to_db(mel_gt,   ref=np.max),
                             sr=sr, hop_length=256, x_axis='time', y_axis='mel', ax=ax_gt_m)
    librosa.display.specshow(librosa.power_to_db(mel_pred, ref=np.max),
                             sr=sr, hop_length=256, x_axis='time', y_axis='mel', ax=ax_pred_m)
    ax_gt_m.set_title('Ground Truth — mel', fontsize=9)
    ax_pred_m.set_title('Reconstruction — mel', fontsize=9)

    score_str = '  '.join([f"{k}: {v:.3f}" for k, v in r['scores'].items()])
    rank = next((i for i, x in enumerate(results) if x['sample_idx'] == sample_idx), '?')
    fig.suptitle(
        f"sample={sample_idx:04d} | sound={r['sound_idx']:04d} | "
        f"rank=#{rank+1 if isinstance(rank,int) else rank} | avg={r['avg_score']:.4f} | {score_str}",
        fontsize=9
    )
    plt.tight_layout()
    plt.show()

    display(HTML(f"<b>▶ Ground Truth</b> &nbsp;<code>{os.path.basename(r['gt_path'])}</code>"))
    display(Audio(gt_wav,   rate=sr))
    display(HTML(f"<b>▶ Reconstruction</b> &nbsp;<code>{os.path.basename(r['pred_path'])}</code>"))
    display(Audio(pred_wav, rate=sr))


available_samples = sorted(set(r['sample_idx'] for r in results))
print(f"Sample index available ({len(available_samples)}): {available_samples}")
print()
plot_and_play(SAMPLE_IDX)